In [ ]:
import sys
sys.path.append("..")#This line tells Python "also look inside this folder when I try to import something."

In [ ]:
import yfinance as yf
tickers= [
    "^GSPC", "^DJI", "^IXIC", "^VIX",
    "AAPL","JPM","XOM","JNJ",
    "^NSEI",
    "TCS.NS","HDFCBANK.NS","RELIANCE.NS","SUNPHARMA.NS"
]
data=yf.download(tickers, period="6mo", interval="1d")
data

In [ ]:
close_prices=data['Close']
close_prices

In [ ]:
#now we are handling the NaN values here using forward fill
close_prices_filled = close_prices.ffill()
close_prices_filled


In [ ]:
close_prices_filled.head(3)
close_prices_filled.isna().sum()

In [ ]:
close_prices_filled.to_csv("../data/processed/close_prices_clean.csv")

In [ ]:
close_prices.to_csv("../data/raw/close_prices_raw.csv")

In [ ]:
#Daily return for a given day = (today's price − yesterday's price) / yesterday's price
# pct_change() compares 1 row back
daily_returns=close_prices_filled.pct_change()
daily_returns

In [ ]:
# checking the above result for one of the column


In [ ]:
# Rolling volatility : Pandas has a .rolling(window_size) method that sets up a rolling window
# After .rolling(), you chain another method to say what to calculate over that window (we want standard deviation)
# A "rolling" calculation means: for each row, look back at the last N rows (including itself) and compute something over just that window

rolling_vol_20d = daily_returns.rolling(window=20).std()
rolling_vol_20d

In [ ]:
#Annualizing Volatility : Finance convention is to annualize volatility so it's comparable to how it's normally quoted (e.g., "this stock has ~25% annual volatility").
# Volatility scales with the square root of time (this comes from statistics — variances add up over independent periods, and standard deviation is the square root of variance). There are roughly 252 trading days in a year, so:
# annualized_volatility = daily_volatility × √252
annualzed_vol_20d = rolling_vol_20d * (252**0.5)
annualzed_vol_20d

In [ ]:
# Sector Rotation
# First assigning sectors to tickers
sector_map={
    "AAPL": "Tech",
    "TCS.NS":"Tech",
    "JPM":"Finance",
    "HDFCBANK.NS":"Finance",
    "XOM":"Energy",
    "RELIANCE.NS":"Energy",
    "JNJ":"Healthcare",
    "SUNPHARMA.NS":"Healthcare"
}
#now we need the pct_change for the last 20 days to find the performance of each ticker
# and then map it sector wise to se the best perfornming sectors
returns_20d=close_prices_filled.pct_change(periods=20)
returns_20d
# we want only the last row of this as it tells between today and 20 days back
latest_20d_returns=returns_20d.tail(1).T #for the last row and then it's transpose
latest_20d_returns.columns=["return_20d"]
latest_20d_returns
# attaching the sector labels to this
latest_20d_returns["sector"]=latest_20d_returns.index.map(sector_map)
latest_20d_returns
# ranking the sectors
sector_ranking=latest_20d_returns.dropna(subset=["sector"])#drops the NaN value rows
sector_ranking=sector_ranking.sort_values("return_20d",ascending=False)
sector_ranking # tickers of the same sector are scattered in this table

In [ ]:
#correlation analysis
correlation_matrix=daily_returns.corr()
correlation_matrix

In [ ]:
#Regime detection using VIX
def classify_regime(vix_level):
    if vix_level<15:
        return "CALM"
    elif vix_level<25:
        return "NORMAL"
    else:
        return "VOLATILE"

close_prices_filled["regime"]=close_prices_filled["^VIX"].apply(classify_regime)
close_prices_filled

In [ ]:
#verify the above
close_prices_filled[["^VIX","regime"]].sort_values("^VIX").head(3)
close_prices_filled[["^VIX","regime"]].sort_values("^VIX").tail(3)

In [ ]:

from src.data_loader import fetch_price_data

In [ ]:
tickers=["^GSPC", "^DJI", "^IXIC", "^VIX",
    "AAPL", "JPM", "XOM", "JNJ",
    "^NSEI", "TCS.NS", "HDFCBANK.NS", "RELIANCE.NS", "SUNPHARMA.NS"]

test_data=fetch_price_data(tickers)
test_data

In [ ]:
from src.data_loader import clean_price_data
clean_data=clean_price_data(test_data)
clean_data

In [ ]:
from src.analysis import calculate_volatility
vol_test=calculate_volatility(clean_data)
vol_test.tail()

In [ ]:
clean_data.to_csv("../data/processed/close_prices_clean.csv")

In [ ]:
import importlib
import src.analysis
importlib.reload(src.analysis)

from src.analysis import calculate_sector_ranking

In [ ]:

ranking_test= calculate_sector_ranking(clean_data, sector_map)
ranking_test

In [ ]:
from src.analysis import calculate_correlation
corr_test=calculate_correlation(clean_data)
corr_test

In [ ]:
importlib.reload(src.analysis)
from src.analysis import classify_regime
clean_data["regime"] = clean_data["^VIX"].apply(classify_regime)
clean_data[["^VIX","regime"]].tail()

In [ ]:
import mysql.connector
#reshape from wide to long format
long_data=clean_data.drop(columns=["regime"]).reset_index()
long_data=long_data.melt(id_vars="Date", var_name="ticker",value_name="close_price")
long_data=long_data.rename(columns={"Date":"date"})
long_data = long_data.dropna(subset=["close_price"])
#connect to MuSQL
conn=mysql.connector.connect(
    host="localhost",
    user="root",
    password="Ahilya23$",
    database="market_monitor"
)
cursor=conn.cursor()
#insert price_history rows
insert_query="""
INSERT INTO price_history(date,ticker,close_price)
VALUES(%s,%s,%s)
ON DUPLICATE KEY UPDATE close_price = VALUES(close_price)
"""
data_tuples=list(long_data.itertuples(index=False,name=None))
cursor.executemany(insert_query,data_tuples)
conn.commit()

#insert ticker_sectors
sector_query="""
    INSERT INTO ticker_sectors(ticker,sector)
    VALUES(%s,%s)
    ON DUPLICATE KEY UPDATE SECTOR = VALUES(sector)
"""
sector_tuples=list(sector_map.items())
cursor.executemany(sector_query,sector_tuples)
conn.commit()

cursor.close()
conn.close()
print("Data loaded successfully")
    


In [ ]:
from src.db import load_price_data,load_sector_data
load_price_data(long_data)
load_sector_data(sector_map)

In [ ]:
from src.db import load_sector_data
load_sector_data(sector_map)

In [ ]:
corr_test.to_csv("../data/processed/correlation.csv")

In [30]:
from src.commentary import generate_market_commentary
latest_regime = clean_data["regime"].iloc[-1] if "regime" in clean_data.columns else classify_regime(clean_data["^VIX"].iloc[-1])
commentary = generate_market_commentary(ranking_test, latest_regime, corr_test)
print(commentary)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


While broader equity markets remain anchored in a low-volatility regime, stark performance dispersion is emerging beneath the surface within the technology sector. Investors are heavily favoring US mega-cap hardware leaders like Apple, even as global IT service providers such as TCS face notable headwinds over the trailing three-week period. This growing intra-sector decoupling highlights a stock-picker's market where idiosyncratic company fundamentals, rather than broad thematic momentum, are dictating performance.


In [31]:
with open("../outputs/market_commentary.txt", "w", encoding="utf-8") as f:
    f.write(commentary)